In [1]:
from ecostyles import EcoStyles
styles = EcoStyles()
styles.register_and_enable_theme(theme_name='article')

import pandas as pd
import altair as alt

import os
os.makedirs('data', exist_ok=True)

# alt.data_transformers.disable_max_rows()

Image

In [9]:
xmax=9
mo=0.5
logo=alt.Chart(pd.DataFrame([{"x": xmax, "img": "https://raw.githubusercontent.com/EconomicsObservatory/ECOvisualisations/main/guidelines/logos/eco-icon-dark_cropped.png"}]))\
    .mark_image(width=40,height=40,align='right',baseline='top',yOffset=-33,opacity=mo,xOffset=0).encode(x=alt.value('width'),url='img:N')

logo

alt.Chart(...)

In [30]:
pub_date = '2026-04-21'
title = 'How is the conflict in the Middle East affecting developing economies?'
title_fmt = pub_date + '-' + title.lower().replace('?', '').replace(' ', '-').replace(':', '')
print(title_fmt)

2026-04-21-how-is-the-conflict-in-the-middle-east-affecting-developing-economies


---

### Figure 1.  Annual FAO Food Price Index

Source:
- https://www.fao.org/worldfoodsituation/foodpricesindex/en/

From Author data file

In [4]:
df = pd.read_excel('raw/Fig 1_FFPI.xlsx', dtype={'Year': str, 'Food Price Index': float})
df.columns = ['date', 'value']
df.tail()

,date,value
32,2022,144.509866
33,2023,124.519666
34,2024,121.981996
35,2025,127.191617
36,2026,126.016715


From FAO data

**1a. Annual**

In [28]:
annual = pd.read_excel('raw/ffpi-data-2026-04.xlsx', sheet_name='Annual')
print(f"Dataset: {annual.columns[0]}")
print(f"Index: {annual.iloc[0,0]}")

annual.columns = annual.loc[1]
annual = annual.iloc[3:].copy()
annual['Year'] = annual['Year'].astype(str)

# Keep first 7 columns
annual = annual.iloc[:, :7].copy()
annual = annual.rename(columns={'Year': 'date', 'Food Price Index': 'All Food'})
annual = pd.melt(annual, id_vars=['date'], var_name='index', value_name='value')
annual['index'] = annual['index'].str.replace(' Price Index', '')
annual['value'] = annual['value'].round(2)
annual

Dataset: Annual FAO Food Price Indices
Index: 2014-2016=100


,date,index,value
0,1991,All Food,62.35
1,1992,All Food,64.23
2,1993,All Food,62.26
3,1994,All Food,67.26
4,1995,All Food,76.83
...,...,...,...
211,2022,Sugar,114.46
212,2023,Sugar,144.98
213,2024,Sugar,125.71
214,2025,Sugar,104.31


Annual: single line

In [6]:
styles.eco_colours

{'red': '#e6224b',
 'blue-light': '#179fdb',
 'blue-dark': '#122b39',
 'yellow': '#f4c245',
 'orange': '#eb5c2e',
 'turquoise': '#36b7b4'}

,date,index,value
0,1991,All Food,62.345284
1,1992,All Food,64.225559
2,1993,All Food,62.259405
3,1994,All Food,67.260547
4,1995,All Food,76.828785
...,...,...,...
211,2022,Sugar,114.455796
212,2023,Sugar,144.980311
213,2024,Sugar,125.712923
214,2025,Sugar,104.307145


In [31]:
data_file = 'fig1_annual.csv'

annual.to_csv('data/' + data_file, index=False)


base_url = 'https://raw.githubusercontent.com/EconomicsObservatory/ECOvisualisations/refs/heads/main/articles/'

c1_data_url = base_url + title_fmt + '/data/' + data_file
print(c1_data_url)

https://raw.githubusercontent.com/EconomicsObservatory/ECOvisualisations/refs/heads/main/articles/2026-04-21-how-is-the-conflict-in-the-middle-east-affecting-developing-economies/data/fig1_annual.csv


In [ ]:
base = alt.Chart(c1_data_url).transform_filter(
    "datum.index == 'All Food'"
).encode(
    alt.X('date:T'),
    alt.Y('value:Q').title('Price index, 2014-2016 = 100').scale(zero=False, padding=30)
).properties(
    width=400,
    height=270
)


lines = base.mark_line(
    interpolate='linear',
    strokeWidth=2.5
)
point = base.mark_point(size=60).encode(
    alt.X('date:T').aggregate('max'),
    alt.Y('value:Q').aggregate({'argmax': 'date'})
)

label1 = base.mark_text(
    align='left',
    dx=7,
    dy=-11
).encode(
    alt.X('date:T').aggregate('max'),
    alt.Y('value:Q').aggregate({'argmax': 'date'}),
    text=alt.value("Jan-Mar 2026")
)
label2 = base.mark_text(
    align='left',
    dx=7,
    dy=0
).encode(
    alt.X('date:T').aggregate('max'),
    alt.Y('value:Q').aggregate({'argmax': 'date'}),
    alt.Text('value:Q').aggregate({'argmax': 'date'}).format('.0f')
)

# Add annotation for Russia-Ukraine war
df_annotation = pd.DataFrame([{"date": "2022-01-24", "text": "Russia invades |Ukraine"}])
vline = alt.Chart(df_annotation).mark_rule(
    color='#676A8680',
    strokeWidth=2, strokeDash=[4, 1]
).encode(
    alt.X('date:T')
)
vline_text = vline.mark_text(
    align='right',
    dx=-5,
    lineBreak='|',
    lineHeight=10
).encode(
    y=alt.value(10),
    text=alt.Text('text:N')
)


chart = vline + vline_text + lines + label1 + label2 + point + logo
chart.display()
styles.save(chart, 'visualisation', 'fig1_annual', width=400, height=270)

alt.LayerChart(...)

Annual: all lines

In [44]:
base = alt.Chart(annual).encode(
    alt.X('date:T'),
    alt.Y('value:Q').title('Price index, 2014-2016 = 100'),
    alt.Color('index:N').legend(None)
).properties(
    width=450,
    height=320
)


lines = base.mark_line(
    interpolate='basis',
    strokeWidth=alt.expr("datum.index == 'All Food' ? 3 : 1.1")
)

labels = base.mark_text(
    align='left',
    dx=5,
    dy=alt.expr("{'All food': 2}[datum.index]")
).encode(
    alt.X('date:T').aggregate('max'),
    alt.Y('value:Q').aggregate({'argmax': 'date'}),
    alt.Text('index:N').aggregate({'argmax': 'date'})
)
lines + labels

alt.LayerChart(...)

In [30]:
df = pd.read_excel('raw/ffpi-data-2026-04.xlsx', sheet_name='Indices_Monthly_Nominal')
print(f"Dataset: {df.columns[0]}")
print(f"Index: {df.iloc[0,0]}")

df.columns = df.loc[1]
df = df.iloc[3:].copy()
df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')

# Keep first 7 columns
df = df.iloc[:, :7].copy()
df = df.rename(columns={'Date': 'date', 'Food Price Index': 'All Food'})
df = pd.melt(df, id_vars=['date'], var_name='index', value_name='value')
df

Dataset: FAO Food Price Index
Index: 2014-2016=100


,date,index,value
0,1990-01-01,All Food,64.444201
1,1990-02-01,All Food,64.728299
2,1990-03-01,All Food,64.033021
3,1990-04-01,All Food,66.015997
4,1990-05-01,All Food,64.632844
...,...,...,...
2605,2025-11-01,Sugar,88.583331
2606,2025-12-01,Sugar,90.702075
2607,2026-01-01,Sugar,89.83755
2608,2026-02-01,Sugar,86.185256


In [36]:
base = alt.Chart(df).encode(
    alt.X('date:T'),
    alt.Y('value:Q').title('Price index, 2014-2016 = 100'),
    alt.Color('index:N').legend(None)
).properties(
    width=450,
    height=320
)


lines = base.mark_line(
    interpolate='basis',
    strokeWidth=alt.expr("datum.index == 'All Food' ? 3 : 1.1")
)

labels = base.mark_text(
    align='left',
    dx=5,
    dy=alt.expr("{'All food': 2}[datum.index]")
).encode(
    alt.X('date:T').aggregate('max'),
    alt.Y('value:Q').aggregate({'argmax': 'date'}),
    alt.Text('index:N').aggregate({'argmax': 'date'})
)
lines + labels

alt.LayerChart(...)